# 04 — The Outer Loop: tuning $\lambda$ automatically

`solver.evaluate(lambdas)` is the toolkit's one primitive for meta-optimization. It returns `EvalMetrics`, which deliberately separates the three signals that raw QPU energy conflates:

| field | meaning |
|---|---|
| `true_objective` | real cost of the best solution — no penalties |
| `violations` | per-constraint magnitudes (continuous, not boolean) |
| `feasible_fraction` | how often the QPU lands in the legal region |

A tuner that only saw energy could not tell *cheap-but-illegal* from *legal-but-expensive*.

In [ ]:
import sys; sys.path.insert(0, "../..")
from problems.vrp import VehicleRoutingProblem
from qlkit import LogisticsSolver, GreedyAssignmentRepair

problem = VehicleRoutingProblem.small_instance()
solver = LogisticsSolver.from_config(problem, "../../configs/local.json",
                                     repair=GreedyAssignmentRepair())

metrics = solver.evaluate({"one_hot": 12.0, "capacity": 2.0})
print(metrics)
print("scalarized:", metrics.scalarize(violation_weight=100.0, infeasibility_weight=5.0))

In [ ]:
# Baseline: random search (this is what you must beat).
import random
rng = random.Random(42)
best = (None, float("inf"))
for i in range(10):
    lambdas = {"one_hot": rng.uniform(1, 25), "capacity": rng.uniform(0.1, 10)}
    score = solver.evaluate(lambdas).scalarize(100.0, 5.0)
    if score < best[1]: best = (lambdas, score)
    print(f"[{i}] {lambdas} -> {score:.2f}")
print("\nwinner:", best)

## Your turn

Replace random search in `starter_kit/my_tuner.py` with something smarter — `skopt.gp_minimize`, Optuna, CMA-ES, or an RL policy. Two hints:

1. **Pipeline your evaluations**: `solver.submit()` + `solver.gather()` lets the dispatcher batch several lambda candidates into one QPU job — a Bayesian tuner with batch acquisition (`q-EI`) exploits this directly.
2. On hardware, every `evaluate()` costs queue time. Budget accordingly (`SolverConfig.max_qpu_jobs` is the client-side circuit breaker).

In [ ]:
# Pipelined evaluation of 3 lambda candidates -> the dispatcher batches them:
handles = [solver.submit({"one_hot": lam, "capacity": 2.0}) for lam in (5.0, 10.0, 15.0)]
for result in solver.gather(*handles):
    print(result.lambdas, "->", result.metrics.true_objective)
solver.close()